# 第 8 章: Survived データの探索と可視化

クラスの偏り、性別・客室クラスと生存率の関係、木の深さによる正解率の変化、混同行列を確認する。

In [ ]:
import sys

sys.path.append("..")

import pandas as pd
import seaborn as sns
from japanese_font import use_japanese_font
from sklearn.metrics import ConfusionMatrixDisplay

from lib.chapter02.iris_preprocessing import split_train_test
from lib.chapter08.survived_classifier import (
    build_pipeline,
    load_survived,
    split_features_and_target,
)
from lib.dataset import data_dir

use_japanese_font();

In [ ]:
df = load_survived(data_dir() / "Survived.csv")
df["Survived"].value_counts().plot.bar(title="生存（1）と死亡（0）の人数");

In [ ]:
survival_rate = df.pivot_table(index="Pclass", columns="Sex", values="Survived")
survival_rate.round(3)

In [ ]:
sns.barplot(data=df, x="Pclass", y="Survived", hue="Sex", errorbar=None);

In [ ]:
x, t = split_features_and_target(df)
split = split_train_test(x, t, test_size=0.2, seed=0)
rows = []
for class_weight in [None, "balanced"]:
    for depth in range(1, 11):
        pipeline = build_pipeline(max_depth=depth, class_weight=class_weight)
        pipeline.fit(split.x_train, split.t_train)
        rows.append(
            {
                "class_weight": str(class_weight),
                "depth": depth,
                "train": pipeline.score(split.x_train, split.t_train),
                "test": pipeline.score(split.x_test, split.t_test),
            }
        )
scores = pd.DataFrame(rows)
scores.pivot(index="depth", columns="class_weight", values=["train", "test"]).round(3)

In [ ]:
balanced = scores[scores["class_weight"] == "balanced"]
balanced.plot(x="depth", y=["train", "test"], title="木の深さと正解率");

In [ ]:
pipeline = build_pipeline(max_depth=5, class_weight="balanced")
pipeline.fit(split.x_train, split.t_train)
ConfusionMatrixDisplay.from_estimator(pipeline, split.x_test, split.t_test);

In [ ]:
model = pipeline.named_steps["model"]
columns = pipeline[:-1].transform(split.x_train).columns
importances = pd.Series(model.feature_importances_, index=columns)
importances.sort_values(ascending=False).round(3)